In [4]:
!pip install wandb

In [8]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env
from gym_trading_env.wrapper import DiscreteActionsWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import pandas_ta as ta
import wandb
from wandb.integration.sb3 import WandbCallback

def preprocess(df):
    df["log_ret"] = np.log(df["close"]).diff()

    # Tendance
    df.ta.macd(append=True)
    df.ta.ema(length=20, append=True)
    df.ta.ema(length=50, append=True)
    
    # Calcul de distance par rapport aux EMA (normalisation)
    df["dist_ema20"] = (df["close"] - df["EMA_20"]) / df["EMA_20"]
    
    # Oscillateurs / Momentum
    df.ta.rsi(length=14, append=True)
    df["RSI_14"] = df["RSI_14"] / 100.0 

    # Volatilité
    df.ta.atr(length=14, append=True)
    df["ATR_14_norm"] = df["ATRr_14"] / df["close"] 

    # Nettoyage
    df.dropna(inplace=True) 
    return df

def reward_function(history):
    # Log return du portefeuille
    return np.log(history["portfolio_valuation", -1] / history["portfolio_valuation", -2])

def metric_portfolio_valuation(history):
    return round(history['portfolio_valuation', -1], 2)

# Initialisation de WandB (Change 'mon_projet' et 'entity' si besoin)
run = wandb.init(
    project="reinforcement-learning-project", 
    config={
        "algo": "PPO",
        "policy_type": "MlpPolicy",
        "total_timesteps": 250000,
    },
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=True,
)

# Création de l'environnement
base_env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess,
    portfolio_initial_value=1_000,
    trading_fees=0.1/100,
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_function,
)

base_env.add_metric('Portfolio Valuation', metric_portfolio_valuation)

#base_env = Monitor(base_env)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])
env = Monitor(base_env)
#env = DummyVecEnv([lambda: env])

# Modèle PPO
model = PPO(
    "MlpPolicy", 
    env, 
    verbose=1, 
    n_steps=2048,
    ent_coef=0.01,
    tensorboard_log=f"runs/{run.id}"
)

print("Début de l'entraînement...")

model.learn(
    total_timesteps=wandb.config.total_timesteps, 
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    ),
)

portfolio_valuation = base_env.historical_info['portfolio_valuation', -1]

run.summary['portfolio_valuation'] = portfolio_valuation
model.save("ppo_trading_final")
evaluate_policy(model, env, n_eval_episodes=10, render=True)
wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env in a DummyVecEnv.
Début de l'entraînement...
Logging to runs/t7jkwh1p/PPO_1
-----------------------------
| time/              |      |
|    fps             | 1029 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 864         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.003172059 |
|    clip_fraction        | 0.0132      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.4        |
|    explained_variance   | -1.64       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0102     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00274    |
|    std

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Market Return : 398.21%   |   Portfolio Return : -244.49%   |   Portfolio Valuation : -1444.93   |   
Market Return : 104.07%   |   Portfolio Return : 295.41%   |   Portfolio Valuation : 3954.12   |   
Market Return : 39.26%   |   Portfolio Return : 92.69%   |   Portfolio Valuation : 1926.9   |   
Market Return : 42.46%   |   Portfolio Return : 99.16%   |   Portfolio Valuation : 1991.63   |   
Market Return :  9.68%   |   Portfolio Return : 18.92%   |   Portfolio Valuation : 1189.17   |   
Market Return : 102.68%   |   Portfolio Return : -101.95%   |   Portfolio Valuation : -19.54   |   
Market Return : 625.64%   |   Portfolio Return : -354.27%   |   Portfolio Valuation : -2542.72   |   
Market Return :  5.73%   |   Portfolio Return : 11.17%   |   Portfolio Valuation : 1111.71   |   
Market Return : 26.64%   |   Portfolio Return : 57.16%   |   Portfolio Valuation : 1571.61   |   
Market Return : 26.64%   |   Portfolio Return : 57.16%   |   Portfolio Valuation : 1571.56   |   


global_step,▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
rollout/ep_len_mean,█▄▄▄▂▁▁▁▁▁▁▁▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
rollout/ep_rew_mean,▁▅▅▅▆▇▇▇▇███████████████████████████████
time/fps,█▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/approx_kl,▂▃▃▄▄█▆▃▂▅▂▂▆▂▃▁▄▂▃▅▄▁▂▁▂▄▆▆▄▄▄▂▂▂▃▃▃▆▂▂
train/clip_fraction,▁▃▂▂▁▁█▂▁▁▂▂▂▄▂▂▁▁▃▂▁▁▁▁▁▃▂▂▂▁▂▁▂▃▅▃▂▄▃▃
train/clip_range,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/entropy_loss,▁▁▁▂▂▂▂▄▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇████▇▇▇▇█
train/explained_variance,▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▁▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇▇
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+4,...
